# 08 Managed Agents 互動實驗室

**說明：** 請先閱讀同目錄的 `README.md` 和各 `.py` 檔案，再開始執行這個 notebook。

本 notebook 涵蓋：
1. Session 管理基礎（01_quickstart.py）
2. 自定義工具呼叫（02_custom_tools.py）
3. Memory Store 三策略比較（03_memory_stores.py）
4. 多 Store Session 協同（04_multi_store_session.py）
5. SSE 串流處理（05_event_streaming.py）
6. 材料科學抽取端對端（06_material_extraction_demo.py）
7. 練習題

> 設定好 `.env` 中的 `ANTHROPIC_API_KEY` 後開始執行（未設定時自動切換 Mock 模式）

In [ ]:
# ── [Cell 0] 環境設定 ──────────────────────────────────────────────────────────
import os
import sys
import json

sys.path.insert(0, '..')  # 讓 import 找到 08_managed_agents/ 目錄

from dotenv import load_dotenv
load_dotenv('../../../.env')  # 載入專案根目錄的 .env

MOCK_MODE = not bool(os.getenv('ANTHROPIC_API_KEY'))
print('環境設定完成')
print(f'模式：{"Mock（無 API key）" if MOCK_MODE else "真實 API"}')
print(f'API key 前綴：{os.getenv("ANTHROPIC_API_KEY", "未設定")[:12]}...' if not MOCK_MODE else '')

---

## Part 1：Session 管理基礎

`AgentSession` 是最小的對話狀態容器。
理解它的結構是掌握 Managed Agent 的第一步。

In [ ]:
# ── [Cell 1] Session 結構探索 ──────────────────────────────────────────────────
from quickstart import AgentSession, chat_turn

# 建立 Session 並查看初始狀態
session = AgentSession(session_id='lab-001')
print('初始狀態：', session.to_summary())

# 手動加入幾條訊息
session.add_user_message('VO₂ 的相變溫度是多少？')
session.add_assistant_message('VO₂ 的半導體—金屬相變溫度約為 68°C（341 K）。')
session.add_user_message('電阻變化幅度呢？')
session.add_assistant_message('電阻可降低約 4 個數量級（ΔR/R ≈ 10⁴）。')

print('\n加入 2 輪對話後：', session.to_summary())
print('\nmessages 結構：')
for m in session.messages:
    print(f"  [{m['role']}] {m['content'][:50]}...")

In [ ]:
# ── [Cell 2] 執行三輪對話 ──────────────────────────────────────────────────────
# TODO: 嘗試修改以下問題，觀察 Session 是否能參考先前的對話

if MOCK_MODE:
    client = None
else:
    from anthropic import Anthropic
    client = Anthropic()

session2 = AgentSession(session_id='lab-002')
questions = [
    '你好，請介紹 VO₂ 的相變特性。',
    '承上，這個特性如何應用於建築節能？',  # 測試上下文參考
    '如果要製備這種薄膜，主要的挑戰是什麼？',
]

for q in questions:
    print(f'\n問：{q}')
    answer = chat_turn(session2, q, client, MOCK_MODE)
    print(f'答：{answer[:150]}...' if len(answer) > 150 else f'答：{answer}')

print(f'\n最終 Session 狀態：', session2.to_summary())

---

## Part 2：自定義工具呼叫

觀察 `tool_use → tool_result` 的完整迴圈，
以及工具 schema 設計如何影響 Agent 的工具選擇決策。

In [ ]:
# ── [Cell 3] 工具呼叫觀察 ──────────────────────────────────────────────────────
from custom_tools import (
    run_agent_with_tools,
    lookup_material_property,
    calculate_film_thickness,
    parse_synthesis_conditions,
)

# 先直接測試本地工具函式
print('=== 本地工具測試 ===')
r1 = lookup_material_property('VO2', 'transition_temperature')
print('lookup_material_property:', json.dumps(r1, ensure_ascii=False))

r2 = calculate_film_thickness(rate=2.0, rate_unit='nm_per_min', duration=50)
print('calculate_film_thickness:', json.dumps(r2, ensure_ascii=False))

r3 = parse_synthesis_conditions(
    'The VO₂ films were deposited at 500°C with 5 mTorr for 50 min, Ar:O₂ ratio 15:1.'
)
print('parse_synthesis_conditions:', json.dumps(r3, ensure_ascii=False))

In [ ]:
# ── [Cell 4] Agent 驅動工具呼叫 ────────────────────────────────────────────────
# TODO: 嘗試不同的問題，觀察 Agent 選擇哪個工具

question = '如果以 2 nm/min 的速率沉積 50 分鐘，薄膜厚度是多少？用 Å 表示也算出來。'
print(f'問題：{question}\n')
answer = run_agent_with_tools(question, client, MOCK_MODE, verbose=True)
print(f'\n最終回答：{answer}')

---

## Part 3：Memory Store 策略比較

比較三種記憶策略在相同對話歷史下的 token 消耗與結構差異。

In [ ]:
# ── [Cell 5] 三種記憶策略統計 ──────────────────────────────────────────────────
from memory_stores import FullHistoryStore, FileBasedStore, SummarisationStore
from pathlib import Path

SAMPLE_CONVERSATION = [
    ('user', 'VO₂ 的相變溫度是多少？'),
    ('assistant', 'VO₂ 的半導體—金屬相變溫度約為 68°C（341 K）。'),
    ('user', '電阻變化幅度呢？'),
    ('assistant', '相變時電阻可降低約 4 個數量級（ΔR/R ≈ 10⁴），在 10⁻³ 到 10⁻² Ω·cm 之間。'),
    ('user', '磁控濺射的典型條件？'),
    ('assistant', '常見條件：工作壓力 5 mTorr、Ar/O₂ 流量比 15:1、基板溫度 500°C、沉積速率 2 nm/min。'),
]

# 初始化三種策略
full = FullHistoryStore()
file_s = FileBasedStore(Path('/tmp/lab_session.jsonl'))
file_s.clear()
summ = SummarisationStore(token_budget=50)  # 低閾值以觸發壓縮

for role, content in SAMPLE_CONVERSATION:
    full.append(role, content)
    file_s.append(role, content)
    summ.append(role, content)

# 觸發壓縮
compressed = summ.maybe_compress(client, MOCK_MODE)

print('策略一（全量）：', json.dumps(full.stats(), ensure_ascii=False, indent=2))
print('\n策略二（檔案）：', json.dumps(file_s.stats(), ensure_ascii=False, indent=2))
print('\n策略三（摘要）：', json.dumps(summ.stats(), ensure_ascii=False, indent=2))
if compressed:
    print(f'\n生成的摘要：{summ.summary}')

In [ ]:
# ── [Cell 6] 觀察策略三的 get_messages() 結構 ─────────────────────────────────
# 摘要壓縮後，messages 結構如何變化？

print('策略三 get_messages() 結構（壓縮後）：')
for i, m in enumerate(summ.get_messages()):
    print(f"  [{i}] role={m['role']}, length={len(m['content'])}, "
          f"preview={m['content'][:60]}...")

---

## Part 4：多 Store Session 協同

觀察工具快取（tool_cache）如何在重複查詢時節省 API 呼叫。

In [ ]:
# ── [Cell 7] 快取命中率測試 ────────────────────────────────────────────────────
from multi_store_session import SessionManager, run_multi_store_agent

session_ms = SessionManager(session_id='lab-multi-001')

questions = [
    '查詢 VO₂ 的相變溫度。',
    '再次確認 VO₂ 的相變溫度（應命中快取）。',  # 測試快取
    '查詢 TiO₂ 的能隙。',
]

for q in questions:
    print(f'\n問：{q}')
    ans = run_multi_store_agent(q, session_ms, client, MOCK_MODE)
    print(f'答：{ans}')

print('\nSession 統計：', json.dumps(session_ms.get_stats(), ensure_ascii=False, indent=2))
print('\nScratchpad：')
for note in session_ms.scratchpad:
    print(f'  {note}')

---

## Part 5：SSE 串流處理

觀察逐 token 輸出效果與事件統計。

In [ ]:
# ── [Cell 8] 串流輸出示範 ──────────────────────────────────────────────────────
from event_streaming import stream_pure_text, stream_with_tools, StreamEventHandler

print('=== 純文字串流 ===')
text_result = stream_pure_text(
    '請用 3 句話描述 VO₂ 的相變特性。',
    client,
    MOCK_MODE,
)
print(f'\n總長度：{len(text_result)} 字')

In [ ]:
# ── [Cell 9] 含工具呼叫的串流 ──────────────────────────────────────────────────
# TODO: 修改問題，觀察串流中工具呼叫的插入時機

print('=== 含工具呼叫的串流 ===')
result = stream_with_tools(
    '請查詢 VO₂ 的相變溫度並說明其工業應用。',
    client,
    MOCK_MODE,
)
print(f'\n串流完成，完整回應長度：{len(result)} 字')

---

## Part 6：材料科學抽取端對端

觀察四步驟流程（初步抽取 → 工具驗證 → 反思修正 → 持久化）。

In [ ]:
# ── [Cell 10] 執行完整抽取流程 ────────────────────────────────────────────────
from material_extraction_demo import (
    run_full_pipeline, SAMPLE_PAPER_TEXT, EXTRACTION_SCHEMA
)

print('論文段落：')
print(SAMPLE_PAPER_TEXT[:200] + '...\n')

result = run_full_pipeline(
    paper_id='arXiv:lab-demo-001',
    paper_text=SAMPLE_PAPER_TEXT,
    client=client,
    mock_mode=MOCK_MODE,
)

In [ ]:
# ── [Cell 11] 抽取結果分析 ────────────────────────────────────────────────────
print('最終抽取結果：')
print(json.dumps(result.final_extraction, ensure_ascii=False, indent=2))

# 計算欄位填充率
total_fields = len(EXTRACTION_SCHEMA)
filled_fields = sum(1 for v in result.final_extraction.values() if v is not None)
print(f'\n欄位填充率：{filled_fields}/{total_fields} = {filled_fields/total_fields:.1%}')
print(f'處理時間：{result.processing_time_s}s')

---

## Part 7：觀察記錄

記錄你的實驗結果和心得：

In [ ]:
# ── [Cell 12] 觀察記錄 ────────────────────────────────────────────────────────

# 請在這裡記錄你的發現：
observations = {
    "session_management": [
        # 例如：「AgentSession 的 messages 在真實 API 模式下，第三輪回應確實參考了第一輪的內容」
        # TODO: 填入你的觀察
    ],
    "tool_use": [
        # 例如：「Agent 在被問到計算問題時，會優先選擇 calculate_film_thickness 而非直接回答」
        # TODO: 填入你的觀察
    ],
    "memory_stores": [
        # 例如：「SummarisationStore 在 token_budget=50 時，3 輪對話後觸發壓縮」
        # TODO: 填入你的觀察
    ],
    "streaming": [
        # 例如：「串流模式下，tool_use block 的 input JSON 是分批傳遞的，需累積後解析」
        # TODO: 填入你的觀察
    ],
    "extraction_quality": [
        # 例如：「論文中的 5 × 10⁻⁷ Torr 是否被正確抽取為浮點數？」
        # TODO: 填入你的觀察
    ],
}

print('觀察記錄已備妥，請填入你的發現。')

---

## Part 8：練習題

請嘗試完成以下練習，加深對 Managed Agents 的理解。

In [ ]:
# ── [練習 1] 自定義新工具 ─────────────────────────────────────────────────────
# 目標：新增一個 `convert_pressure_units` 工具，支援 Pa ↔ Torr ↔ mTorr 換算
# 提示：1 Torr = 133.322 Pa = 1000 mTorr

def convert_pressure_units(value: float, from_unit: str, to_unit: str) -> dict:
    """壓力單位換算工具（練習：請實作）"""
    # TODO: 實作換算邏輯
    pass

EXERCISE_1_TOOL = {
    # TODO: 填入工具的 JSON Schema 定義
    "name": "convert_pressure_units",
    "description": "",  # TODO
    "input_schema": {},  # TODO
}

# 測試（預期：5 mTorr → 0.666 Pa）
# result = convert_pressure_units(5, 'mTorr', 'Pa')
# assert abs(result['value'] - 0.666) < 0.01
print('練習 1：請實作 convert_pressure_units 工具')

In [ ]:
# ── [練習 2] 測試記憶壓縮閾值 ────────────────────────────────────────────────
# 目標：調整 SummarisationStore 的 token_budget，觀察何時觸發壓縮
# 問題：budget=200 vs budget=500，需要幾輪對話才觸發壓縮？

# TODO: 實作實驗，填入觀察結果
for budget in [200, 500]:
    store = SummarisationStore(token_budget=budget)
    for role, content in SAMPLE_CONVERSATION:
        store.append(role, content)
    triggered = store.maybe_compress(client, MOCK_MODE)
    print(f'budget={budget}: 壓縮觸發={triggered}, tokens={store._total_tokens}')

In [ ]:
# ── [練習 3] 自定義論文段落測試 ───────────────────────────────────────────────
# 目標：用你找到的真實材料科學論文段落測試抽取效果
# 來源建議：Google Scholar 搜尋 "thin film deposition VO2" 或 "lithium-ion battery cathode"

YOUR_PAPER_TEXT = """
TODO: 貼上你找到的論文段落（英文或中文均可）
"""

if 'TODO' not in YOUR_PAPER_TEXT:
    result = run_full_pipeline(
        paper_id='my-paper-001',
        paper_text=YOUR_PAPER_TEXT,
        client=client,
        mock_mode=MOCK_MODE,
    )
    print(json.dumps(result.final_extraction, ensure_ascii=False, indent=2))
else:
    print('請先在 YOUR_PAPER_TEXT 中貼入論文段落。')

In [ ]:
# ── [練習 4] 效能基準測試 ─────────────────────────────────────────────────────
# 目標：比較有快取 vs 無快取的工具呼叫次數
# 提示：使用 SessionManager 的 get_stats() 比較 tool_calls 與 cache_hits

import time

# TODO: 建立兩個 SessionManager（一個用快取，一個不用）
# 對相同問題呼叫 3 次，比較統計數據
print('練習 4：請實作快取效能比較實驗')

In [ ]:
# ── [練習 5 — 挑戰題] 整合 ACE Playbook ──────────────────────────────────────
# 目標：從 06_material_extraction_demo 的結果中提取 1 條「教訓」，
# 並思考如何將它格式化為 ACE Playbook 規則注入 System Prompt

# 參考：04_ace_framework/ 的 curator_pattern.py

# TODO: 實作教訓提取邏輯
example_lesson = {
    "rule": "TODO: 填入從本次抽取中學到的規則",
    "trigger": "TODO: 何時適用此規則",
    "example": "TODO: 具體案例",
}
print('練習 5（挑戰）：請填入你從抽取結果中提取的教訓')
print(json.dumps(example_lesson, ensure_ascii=False, indent=2))